# MODFLOW 6 API E: streamflow augmentation at the coast

This is notebook E of the API series. Notebook A ([`mf6-api-a-basic`](mf6-api-a-basic.ipynb)) covers the steps of an API run, and notebook C ([`mf6-api-c-recharge-callback`](mf6-api-c-recharge-callback.ipynb)) covers callbacks.

**Streamflow augmentation** pumps groundwater into a stream to keep its flow above a target in dry years. Run an operating rule on the synthetic-valley advanced model: whenever the flow at the stream gage falls below the target, pump a well and put the pumped water into the top of the stream. Pumping the well also lowers the heads around the stream, so part of the water put into the stream leaks back into the aquifer. How much of the pumped water the stream keeps depends on where else the well can get its water from, and in this valley the sea is close by.

> **Model reference.** The synthetic valley model is described in Hughes, J.D., Langevin, C.D., Paulinski, S.R., Larsen, J.D., and Brakenhoff, D., 2024, FloPy workflows for creating structured and unstructured MODFLOW models: Groundwater, v. 62, no. 1, p. 124–139, https://doi.org/10.1111/gwat.13327.

## What this notebook covers

By the end of this notebook you will be able to:

- represent the weight of seawater in a model of constant fluid density with **equivalent freshwater heads** at the coast,
- read the streamflow from a running model and set a pumping rate from it,
- compare an operating rule updated once per stress period with one updated every outer iteration, and
- measure how much of the pumped water the stream keeps, and check it against the fraction of the well's water that comes from the stream.

## Imports

Import FloPy, NumPy, pandas, and Matplotlib, and `Callbacks` and `run_simulation` from `modflowapi`. `mf6_api_helpers` finds the MODFLOW 6 library, and `ADVANCED_MODEL` from `sv_notebook_helpers` is the folder of the synthetic-valley advanced model.

In [ ]:
%matplotlib inline
import pathlib as pl
import shutil

import flopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mf6_api_helpers import find_mf6_libraries
from modflowapi import Callbacks, run_simulation
from sv_notebook_helpers import ADVANCED_MODEL

## Find the MODFLOW 6 library

The API runs MODFLOW 6 through its shared library, `libmf6` (`libmf6.so` on Linux, `libmf6.dylib` on macOS, and `libmf6.dll` on Windows), rather than through the `mf6` program. pixi installs both with the class MODFLOW 6. `find_mf6_libraries()` returns the path of each.

In [ ]:
lib_path, mf6_exe = find_mf6_libraries()
print(f"library: {lib_path.name}")
print(f"program: {mf6_exe.name}")

## The coast

The sea is a General-Head Boundary (**GHB**) along the southern row of the model, at a head of 0 m, sea level, in every layer. Seawater is denser than fresh groundwater, so at any depth below sea level the sea presses harder on the aquifer than fresh water at sea level would. This model has one fluid density, that of fresh water. To carry the weight of the sea, raise the head of each boundary to the **equivalent freshwater head**, the head of a column of fresh water that presses as hard at that depth as the sea does:

$$h_f = h_s + \left(\frac{\rho_s}{\rho_f} - 1\right)\left(h_s - z\right),$$

where $h_f$ is the equivalent freshwater head (L), $h_s$ is sea level (L), $\rho_s$ and $\rho_f$ are the densities of seawater and fresh water (M/L³), and $z$ is the elevation of the boundary (L). The GHB of the advanced model already lists each boundary's elevation and the salt concentration of seawater, 35 kilograms per cubic meter (kg/m³), as the auxiliary variables `elevation` and `concentration`. The density of water rises by 0.7 kg/m³ for each kg/m³ of salt, so seawater is 1,024.5 kg/m³.

`build()` below loads the advanced model and switches off its prediction well, a well in layer 5 that the operating rule will pump, so the run is a baseline without it. With `coast="freshwater-equivalent"` it raises the GHB heads; with `coast="none"` it removes the GHB, giving the same valley with no sea, whose southern edge lets no water through. With `pumping` it pumps the prediction well at a constant rate from `aug_start`, which is used at the end of the notebook. It writes the simulation and, unless `run=False`, runs it with the `mf6` program.

In [ ]:
name = "sv"
start_date = "1962-01-01"
aug_start = 10  # the rule runs in the stress periods after this one, from 0
FRESHWATER_DENSITY = 1000.0  # kg/m³
DENSITY_SLOPE = 0.7  # rise in density for each kg/m³ of salt


def build(workspace, coast="freshwater-equivalent", pumping=0.0, run=True):
    """Write, and run, the advanced model with the prediction well set."""
    workspace = pl.Path("models") / workspace
    if workspace.exists():
        shutil.rmtree(workspace)
    sim = flopy.mf6.MFSimulation.load(
        sim_ws=ADVANCED_MODEL,
        exe_name=mf6_exe,
        write_headers=False,
        verbosity_level=0,
    )
    gwf = sim.get_model(name)

    # the prediction well is listed in the transient periods; pump it only
    # after aug_start
    prediction = gwf.get_package("prediction")
    rates = {}
    for kper in range(sim.tdis.nper.data):
        period = prediction.stress_period_data.get_data(kper)
        if period is None:
            continue
        period["q"] = -pumping if kper > aug_start else 0.0
        rates[kper] = period
    prediction.stress_period_data.set_data(rates)

    ghb = gwf.get_package("ghb-1")
    if coast == "freshwater-equivalent":
        data = ghb.stress_period_data.get_data(0)
        density = FRESHWATER_DENSITY + DENSITY_SLOPE * data["concentration"]
        ratio = density / FRESHWATER_DENSITY
        data["bhead"] += (ratio - 1.0) * (data["bhead"] - data["elevation"])
        ghb.stress_period_data.set_data({0: data})
    elif coast == "none":
        gwf.remove_package("ghb-1")

    sim.set_sim_path(workspace)
    sim.write_simulation(silent=True)
    if run:
        success, buff = sim.run_simulation(silent=True)
        assert success, "MODFLOW 6 did not terminate normally"
    return sim, workspace


coastal_sim, coastal_ws = build("mf6-api-e-coast")
inland_sim, inland_ws = build("mf6-api-e-no-coast", coast="none")

ghb = coastal_sim.get_model(name).ghb.stress_period_data.get_data(0)
layers = np.array([cellid[0] for cellid in ghb["cellid"]])
print("equivalent freshwater head of the coast")
for k in np.unique(layers):
    rows = ghb[layers == k]
    print(
        f"  layer {k + 1}: {rows['bhead'].min():.3f} to {rows['bhead'].max():.3f} m"
        f" at elevations of {rows['elevation'].max():.1f} to"
        f" {rows['elevation'].min():.1f} m"
    )

**What to look for.** The equivalent freshwater head rises with depth, from 0.019 m in layer 1, where the boundaries are 0.8 m below sea level, to 0.94 to 1.76 m in layer 5, where they are 38 to 72 m below it. Layer 3 has no coastal boundary, because there is no clay along the coast and its cells there are pass-through cells. Every boundary head is above sea level, so the sea holds back groundwater flowing out to it and, where the aquifer head is lower, flows in.

### Map the valley

Map the features the rule works with: the stream, the lake, the coast, the two production wells of the Multi-Aquifer Well (MAW) package, and the prediction well. The background is the water table at the end of the steady-state period. The distances printed below are from the prediction well to the nearest stream reach and to the coastal row.

In [ ]:
width = 17.15 / 2.54  # figure width, inches
gwf = coastal_sim.get_model(name)
xc, yc = gwf.modelgrid.xcellcenters, gwf.modelgrid.ycellcenters
head = gwf.output.head().get_data(kstpkper=(0, 0))
water_table = np.ma.masked_greater(head[0], 1.0e29)

stream = [r["cellid"][1:] for r in gwf.sfr.packagedata.get_data()]
lake = [cellid[1:] for cellid in gwf.lak.connectiondata.get_data()["cellid"]]
coast = sorted({cellid[1:] for cellid in ghb["cellid"]})
wells = sorted({r["cellid"][1:] for r in gwf.pwell.connectiondata.get_data()})
_, pi, pj = gwf.prediction.stress_period_data.get_data(aug_start + 1)["cellid"][0]

with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=(0.6 * width, 0.95 * width), layout="constrained")
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    pc = mm.plot_array(water_table, cmap="Blues_r", alpha=0.6)
    contours = mm.contour_array(
        water_table, levels=np.arange(0.0, 8.0, 1.0), colors="0.3", linewidths=0.6
    )
    ax.clabel(contours, fmt="%.0f", fontsize=7)
    for cells, color, label in (
        (lake, "#2a78d6", "Lake (LAK)"),
        (stream, "#1baf7a", "Stream (SFR)"),
        (coast, "#6250d6", "Sea (GHB)"),
    ):
        ax.plot(
            [xc[i, j] for i, j in cells],
            [yc[i, j] for i, j in cells],
            "s",
            ms=4,
            color=color,
            label=label,
        )
    ax.plot(
        [xc[i, j] for i, j in wells],
        [yc[i, j] for i, j in wells],
        "o",
        ms=7,
        color="black",
        label="Production well (MAW)",
    )
    ax.plot(
        xc[pi, pj],
        yc[pi, pj],
        "*",
        ms=13,
        color="#eb6834",
        mec="black",
        label="Prediction well",
    )
    ax.set_aspect("equal")
    flopy.plot.styles.xlabel(ax=ax, label="x (m)")
    flopy.plot.styles.ylabel(ax=ax, label="y (m)")
    fig.colorbar(pc, ax=ax, shrink=0.4, label="Water table (m)")
    fig.legend(loc="outside lower center", ncols=2, fontsize=8, frameon=False)

to_stream = min(
    np.hypot(xc[i, j] - xc[pi, pj], yc[i, j] - yc[pi, pj]) for i, j in stream
)
to_coast = yc[pi, pj] - yc[coast[0]]
print(
    f"prediction well: {to_stream:,.0f} m from the stream, {to_coast:,.0f} m from the coast"
)

**What to look for.** The prediction well (star) is in layer 5, 1,067 m east of the stream and 762 m north of the coast, so both are within its reach. The `vc` production well is 457 m west of the stream, and the `reilly` well is east of the lake.

## What the coast does before the well is switched on

The stream gage is the SFR observation `riv-flow` in `sv.sfr.obs`, the flow out of reach 17 of 18. Read it from both baselines with `gwf.sfr.output.obs().get_dataframe()`, which labels each value with the date at the end of its stress period. SFR reports a flow leaving a reach as negative, so change its sign. Flows are in cubic meters per day (m³/d).

The target is 40,000 m³/d at the gage. The rule runs from 2016 to 2025, the stress periods after `aug_start`.

In [ ]:
min_flow = 40000.0  # target flow at the gage, m³/d


def gage_flow(sim):
    """Flow at the gage at the end of each stress period, m³/d."""
    observations = sim.get_model(name).sfr.output.obs()
    return -observations.get_dataframe(start_datetime=start_date)["RIV-FLOW"]


coastal_base = gage_flow(coastal_sim)
inland_base = gage_flow(inland_sim)
dates = coastal_base.index
window = slice(aug_start + 1, None)  # the stress periods the rule runs in

for label, series in (("coast", coastal_base), ("no coast", inland_base)):
    during = series.iloc[window]
    print(
        f"{label}: lowest {during.min():,.0f} m³/d, mean {during.mean():,.0f} m³/d,"
        f" below the target in {(during < min_flow).sum()} of {len(during)} years"
    )
difference = (coastal_base - inland_base).mean()
print(f"the coast adds {difference:,.0f} m³/d to the gage on average")

In [ ]:
years = np.array([date.year - 1 for date in dates])  # the year each period covers
rule_start = years[window][0] - 0.5  # the gray band marks the years the rule runs

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(width, 0.45 * width), layout="constrained")
    # the flow at the end of each year, side by side for the two valleys
    ax.bar(years - 0.2, coastal_base, width=0.4, color="#2a78d6", label="Coast")
    ax.bar(years + 0.2, inland_base, width=0.4, color="#eda100", label="No coast")
    ax.axhline(min_flow, color="black", ls=":", lw=1.0, label="Target")
    ax.axvspan(
        rule_start, years[-1] + 0.5, color="0.9", lw=0, zorder=0, label="Rule runs"
    )
    flopy.plot.styles.heading(ax=ax, heading="Gage flow with the prediction well off")
    flopy.plot.styles.xlabel(ax=ax, label="Year")
    flopy.plot.styles.ylabel(ax=ax, label="Flow (m³/d)")
    # the transient years; the first stress period is the steady state
    ax.set_xticks(years[1::2])
    ax.set_xlim(years[1] - 0.5, years[-1] + 0.5)
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.StrMethodFormatter("{x:,.0f}"))
    fig.legend(loc="outside lower center", ncols=4, frameon=False)

**What to look for.** With the prediction well off, the coast raises the gage flow by 1,212 m³/d on average over the 21 stress periods. With no coast, the gage flow swings further from year to year: higher in the years of highest flow, such as 2018, and lower in the years of lowest flow, such as 2025. In the years the rule runs, 2016 to 2025, the gage is below 40,000 m³/d in 7 years with the coast and 8 without it, and its lowest flow is 31,593 m³/d with the coast and 26,292 m³/d without it.

## The operating rule

When the flow at the gage is below `min_flow`, pump the prediction well at a rate proportional to the shortfall, `GAIN` times the shortfall, up to `RATE_CAP`, and add the same rate as inflow to the first reach of the stream. When the gage is at or above the target, switch the well off.

The callback reads the flow out of every reach, the SFR variable `DSFLOW`, with `get_value()`, and takes the reach of the gage, `gage_reach`. It sets the pumping rate through the running Well package, `ml.prediction`, and the inflow to the first reach through `get_value_ptr()` on the SFR variable `INFLOW`, which is the array MODFLOW itself uses, so writing to it changes the model.

`update_each_iteration` chooses how often the rule acts:

- **False** sets the rate once, at the start of each stress period, from the flow at the end of the previous one.
- **True** also resets it at the start of every outer iteration, from the flow in the current solution, so the rate converges with the heads instead of lagging a year behind.

Pumping lowers the flow the rule reads, so a rate reset in full every outer iteration would chase its own effect and never settle. Each outer iteration the rate moves only part of the way, `relax`, toward the rate the current flow calls for.

In [ ]:
gage_reach = 16  # reach 17, counted from 0
GAIN = 2.5  # pumping rate per unit of shortfall
RATE_CAP = 20000.0  # most the well can pump, m³/d
relax = 0.5  # part of the way the rate moves each outer iteration

aug_rate = 0.0  # m³/d, kept between outer iterations
update_each_iteration = False


def callback(sim, step):
    """Pump the prediction well to hold the gage at min_flow."""
    global aug_rate
    ml = sim.sv

    def required_rate():
        # the rate the flow now at the gage calls for
        flow = sim.mf6.get_value(sim.mf6.get_var_address("DSFLOW", "SV", "SFR-1"))
        q = float(flow[gage_reach])
        if q < min_flow:
            return min(GAIN * (min_flow - q), RATE_CAP)
        return 0.0

    def apply_rate(rate):
        # pump the well and put the same rate into the top of the stream
        ml.prediction.stress_period_data["q"] = -rate
        inflow = sim.mf6.get_value_ptr(sim.mf6.get_var_address("INFLOW", "SV", "SFR-1"))
        inflow[0] = rate

    if sim.kper <= aug_start:
        return
    if step == Callbacks.stress_period_start:
        aug_rate = required_rate()
        apply_rate(aug_rate)
    elif step == Callbacks.iteration_start and update_each_iteration:
        aug_rate += relax * (required_rate() - aug_rate)
        aug_rate = min(max(aug_rate, 0.0), RATE_CAP)
        apply_rate(aug_rate)

### Run the rule both ways

Run the coastal model once with each way of updating the rule, each in its own workspace. `run_simulation()` runs MODFLOW 6 through the library and calls `callback` at every point of the run.

`pumping_rate()` reads what the well actually pumped from the cell-by-cell budget file, `sv.cbc`, rather than taking the rate the callback asked for, which checks that the rule did what it meant to.

In [ ]:
def pumping_rate(sim):
    """Pumping rate of the prediction well in each stress period, m³/d."""
    budget = sim.get_model(name).output.budget()
    rates = []
    for totim in budget.get_times():
        record = budget.get_data(totim=totim, paknam2="prediction")[0]
        rates.append(0.0 if len(record) == 0 else -float(record["q"][0]))
    return np.array(rates)


def run_rule(workspace, coast, per_iteration):
    """Run the model under the rule; return the gage flow and pumping rate."""
    global aug_rate, update_each_iteration
    sim, ws = build(workspace, coast=coast, run=False)
    aug_rate = 0.0
    update_each_iteration = per_iteration
    run_simulation(lib_path, ws, callback, verbose=False)
    return gage_flow(sim), pumping_rate(sim)


updates = {"Once per stress period": False, "Every outer iteration": True}
coastal = {
    label: run_rule(
        f"mf6-api-e-coast-{per_iteration}", "freshwater-equivalent", per_iteration
    )
    for label, per_iteration in updates.items()
}
for label, (flow, rate) in coastal.items():
    during = flow.iloc[window]
    print(
        f"{label}: gage {during.min():,.0f} to {during.max():,.0f} m³/d,"
        f" below the target in {(during < min_flow).sum()} years;"
        f" pumping 0 to {rate.max():,.0f} m³/d"
    )

In [ ]:
colors = {"Once per stress period": "#eb6834", "Every outer iteration": "#2a78d6"}

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A"], ["B"]], figsize=(width, 0.75 * width), layout="constrained", sharex=True
    )
    # the flow at the end of each year, side by side for the three runs
    ax = axd["A"]
    flows = {"No rule": (coastal_base, "0.6")}
    flows.update({label: (flow, colors[label]) for label, (flow, _) in coastal.items()})
    for offset, (label, (flow, color)) in zip(
        (-0.27, 0.0, 0.27), flows.items(), strict=True
    ):
        ax.bar(years + offset, flow, width=0.27, color=color, label=label)
    ax.axhline(min_flow, color="black", ls=":", lw=1.0, label="Target")
    ax.axvspan(
        rule_start, years[-1] + 0.5, color="0.9", lw=0, zorder=0, label="Rule runs"
    )
    flopy.plot.styles.heading(ax=ax, letter="A", heading="Gage flow")
    flopy.plot.styles.ylabel(ax=ax, label="Flow (m³/d)")
    fig.legend(loc="outside lower center", ncols=5, frameon=False)

    # the rate is held over a stress period, so draw it as a bar for each year
    ax = axd["B"]
    for offset, (label, (_, rate)) in zip((-0.2, 0.2), coastal.items(), strict=True):
        ax.bar(years + offset, rate, width=0.4, color=colors[label])
    ax.axvspan(rule_start, years[-1] + 0.5, color="0.9", lw=0, zorder=0)
    flopy.plot.styles.heading(
        ax=ax, letter="B", heading="Pumping rate of the prediction well"
    )
    flopy.plot.styles.xlabel(ax=ax, label="Year")
    flopy.plot.styles.ylabel(ax=ax, label="Pumping (m³/d)")
    # the transient years; the first stress period is the steady state
    ax.set_xticks(years[1::2])
    ax.set_xlim(years[1] - 0.5, years[-1] + 0.5)
    for ax in axd.values():
        ax.yaxis.set_major_formatter(
            plt.matplotlib.ticker.StrMethodFormatter("{x:,.0f}")
        )

**What to look for.** Updating once per stress period (orange) sets each year's rate from the gage flow of the year before, so it answers last year's shortfall. It pumps nothing in 2021, a dry year, because the gage was above the target in 2020, and then pumps 9,135 m³/d in 2022. It pumps 12,121 m³/d in 2024 and nothing in 2025, the driest year, when the gage falls to 31,362 m³/d, below the 31,593 m³/d with no rule at all, because the heads near the stream are still recovering from the pumping of 2024.

Updating every outer iteration (blue) sets the rate from the flow of the year being solved. It pumps in each of the 7 dry years and never more than 8,553 m³/d, and the lowest gage flow is 36,579 m³/d. It still leaves the gage below the target in those 7 years, because the rate is proportional to the shortfall: the well pumps only while the gage is below the target, so the rule settles a little short of it. A larger `GAIN` brings the gage closer. The two rules pump about the same in total.

## How much of the pumped water does the stream keep?

Pumping the well lowers the heads near the stream, so part of the water put into the top of the stream leaks back into the aquifer. The percentage of the pumped water that the gage gains is what the rule is worth. Measure it as the extra water that passed the gage over the years the rule runs, as a percentage of the water pumped; both are in million cubic meters (million m³). Run the valley with no coast under the same rule to see what the sea does.

In [ ]:
inland = {
    label: run_rule(f"mf6-api-e-no-coast-{per_iteration}", "none", per_iteration)
    for label, per_iteration in updates.items()
}

rows = []
for valley, baseline, results in (
    ("Coast", coastal_base, coastal),
    ("No coast", inland_base, inland),
):
    for label, (flow, rate) in results.items():
        during = flow.iloc[window]
        # yearly rates in m³/d summed over the years, to million m³
        rise = (flow - baseline).iloc[window].sum() * 365.25 / 1.0e6
        pumped = rate[window].sum() * 365.25 / 1.0e6
        rows.append(
            {
                "valley": valley,
                "update": label,
                "lowest gage (m³/d)": during.min(),
                "years below target": int((during < min_flow).sum()),
                "pumped (million m³)": pumped,
                "gage rise (million m³)": rise,
                "kept (percent of pumped)": 100.0 * rise / pumped,
            }
        )
summary = pd.DataFrame(rows).set_index(["valley", "update"])
summary.round(3).round({"kept (percent of pumped)": 1})

**What to look for.** The stream with a coast keeps 57.2 percent of the water pumped when the rule is updated once per stress period, and 57.7 percent when it is updated every outer iteration. With no coast, the stream keeps only 4.4 and 1.4 percent: the rule pumps 35.68 and 40.29 million m³ to raise the gage by 1.57 and 0.55 million m³, and still leaves it below the target in 8 of the 10 years. The same well, rule, and target return more than ten times as much water to the stream once the valley has a coast.

### Check against the stream's share of the pumped water

The water a well pumps comes from less discharge to the stream and the sea, more inflow from them, and less water in storage. The percentage that comes from the stream is the well's **stream capture fraction**. If the well takes back that percentage of the water the rule puts into the stream, the stream should keep about the rest.

Measure the capture fraction directly: pump the prediction well at a constant 10,000 m³/d over the years the rule runs, without putting the water into the stream, and divide the mean fall in the gage flow by the pumping rate to get the percentage.

In [ ]:
test_rate = 10000.0  # m³/d
capture, capture_sims = {}, {}
for valley, coast, baseline in (
    ("Coast", "freshwater-equivalent", coastal_base),
    ("No coast", "none", inland_base),
):
    sim, _ = build(f"mf6-api-e-capture-{coast}", coast=coast, pumping=test_rate)
    fall = (baseline - gage_flow(sim)).iloc[window].mean()
    capture[valley] = 100.0 * fall / test_rate
    capture_sims[valley] = sim

for valley, percent in capture.items():
    kept = summary.loc[valley, "kept (percent of pumped)"]
    print(
        f"{valley}: capture fraction {percent:.1f} percent, so the stream should "
        f"keep {100.0 - percent:.1f} percent of the water pumped; "
        f"the rule kept {kept.min():.1f} to {kept.max():.1f} percent"
    )

**What to look for.** With the coast, 41.7 percent of the water the well pumps comes from the stream, so the stream should keep 58.3 percent of the water the rule puts into it; the rule kept 57.2 to 57.7 percent. With no coast, 97.3 percent of it comes from the stream, so the stream should keep 2.7 percent; the rule kept 1.4 to 4.4 percent. One run at a constant pumping rate predicts what the operating rule is worth before the rule is written.

### Where the well's water comes from

The capture fraction says how much of the well's water comes from the stream. The water budget says where the rest comes from. Take the change in each term of the water budget when the well pumps, averaged over the years the rule runs, and divide it by the pumping rate to get the percent of the pumped water from each source. A positive share is water the well gets from that source, as more inflow or less outflow; the shares add up to 100 percent.

In [ ]:
# budget terms, from the list file, that make up each source
sources = {
    "Sea (coast)": ("ghb",),
    "Stream": ("sfr",),
    "Storage": ("sto",),
    "Lake": ("lak",),
    "Drains": ("drn",),
    "Recharge reaching the water table": ("uzf-gwrch",),
}


def budget_rates(sim):
    """Inflow minus outflow of each budget term in the years the rule runs (m³/d)."""
    listing = sim.get_model(name).output.list()
    rates, _ = listing.get_dataframes(diff=True, start_datetime=None)
    return rates.iloc[window]


shares = {}
for valley, base_sim in (("Coast", coastal_sim), ("No coast", inland_sim)):
    change = (budget_rates(capture_sims[valley]) - budget_rates(base_sim)).mean()
    pumped = -change.pop("wel")
    share = {}
    for source, prefixes in sources.items():
        terms = [term for term in change.index if term.startswith(prefixes)]
        share[source] = 100.0 * change[terms].sum() / pumped
        change = change.drop(terms)
    share["Other"] = 100.0 * change.sum() / pumped
    shares[valley] = share

# the list file holds single-precision numbers
sources_table = pd.DataFrame(shares).astype("float64")
sources_table.index.name = "percent of the water pumped"
sources_table.loc["Total"] = sources_table.sum()
# adding zero turns -0.0 into 0.0
sources_table.round(1) + 0.0

**What to look for.** With the coast, 60.6 percent of the well's water comes from the sea: 33.6 percent as less fresh groundwater flowing out to it and 27.0 percent as more seawater flowing in. Another 42.1 percent comes from the stream. Storage and the lake supply less than 1 percent each. With no coast, the stream supplies 103.2 percent, more than the well pumps, and storage, the lake, and the drains supply 1.2 to 1.8 percent each. In both valleys less recharge reaches the water table while the well pumps, 4.0 percent of the pumping with the coast and 7.8 percent without it, and the other sources make up that loss as well. The stream's share, 42.1 percent, differs a little from the capture fraction, 41.7 percent, measured at the gage; the budget counts every reach, including the one below the gage.

## Recap

You represented the weight of seawater along the coast of the synthetic-valley advanced model with equivalent freshwater heads, and ran an operating rule through a `modflowapi` callback that reads the flow at the stream gage and sets both the pumping rate of a well and the inflow to the top of the stream. Updating the rule every outer iteration held the gage closer to the target in the driest years than updating it once per stress period, for about the same total pumping. With the coast, the stream kept 57 percent of the water pumped, compared with 1 to 4 percent with no coast, because the well takes most of its water from the sea; the share of the well's water that does not come from the stream, from a single run at a constant rate, predicted it.